# Final Push

In [1]:
# debug mode flag
DEBUG_MODE = True

if DEBUG_MODE:
    print(
        f"\033[31m WARNING: DEBUG MODE is ON!!! \033[0m Verbose output and debug logging enabled."
    )


In [2]:
# user configurable settings, there is an additional configurable settings block after the whisperx auto-config portion with model-specific settings such as thread count, batch size, compute datatype, toggle-able diarization, alignment, etc.

# toggle offline mode on/off
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # "1" for on, "0" for off

MODEL_DIR = "models/"

OUTPUT_FILE = "transcript.txt"  # Output file written to the project root for now, to be updated when merged and refactored into main branch
INPUT_FILE = "sample_data/en_US/11_Lecture-48k.wav"  # should accept basically any audio or video file that ffmpeg can detect audio in, mp4 or otherwise, refer to ffmpeg docs for more info


In [3]:
# setup cell

# TODO need to prune unused imports, organize, etc
import logging, time, subprocess, re, sys, json, time, os
from datetime import datetime, timedelta
from pathlib import Path
import whisperx
from faster_whisper import WhisperModel
from pydantic_settings import BaseSettings, SettingsConfigDict
import numpy as np

# basic logging setup, add handler to also log to stdout
logging.basicConfig(level=logging.INFO if not DEBUG_MODE else logging.DEBUG)  # toggle debug output here
logging.info(f"Basic logger setup!")

start_time = time.perf_counter()  # begin timer for determining final execution time at end, also double-purposing it to determine execution time for the setup, config, hardware and software detection, etc steps
logging.info(f"Execution officially starting! Start time: {datetime.now()}")

if os.environ["HF_HUB_OFFLINE"] == "1":
    logging.info(
        # blue text indicating offline mode active
        f"\033[94m Offline mode set. \033[0m Will use local (cached) models only."
    )
else:
    logging.warning(
        # red warning text indicating to user that online mode is ACTIVE, and will require network connection for initial downloading of models to a local cache
        f"\033[31m Warning! Online mode set! \033[0m This will be slow for first run, but once models have been downloaded and cached locally, set `os.environ['HF_HUB_OFFLINE'] = '1'` (see top of file, this env variable is set in the initial user configurable settings chunk) to disable online mode and use only the local (cached) models (network connection/internet access should _not_ be required after that point)."
    )


# setup .env settings obj
class GlobalAppSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="allow",  # `extra` param allows added env vars
    )


settings = GlobalAppSettings()
logging.info(f"GlobalAppSettings setup.")


INFO:root:Basic logger setup!
INFO:root:Execution officially starting! Start time: 2026-03-09 03:56:13.978443
INFO:root: Offline mode set.  Will use local (cached) models only.
INFO:root:GlobalAppSettings setup.


In [4]:
import torch
import torchaudio
import pyannote.audio

if DEBUG_MODE:
    logging.info(f"Checking versions of torch, torchaudio, and pyannote.audio...")
    logging.info(f"{torch.__version__=}")
    logging.info(f"{torchaudio.__version__=}")
    logging.info(f"{pyannote.audio.__version__=}")

# torchcodec is still not playing nicely with the various other libraries, but we've since bypassed the requirement for it utilizing ffmpeg to decode audio, so these warnings and errors are expected and can be safely ignored without issue. Nothing to see here, folks


DEBUG:torio._extension.utils:Loading FFmpeg6
DEBUG:torio._extension.utils:Failed to load FFmpeg6 extension.
Traceback (most recent call last):
  File "c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv312\Lib\site-packages\torio\_extension\utils.py", line 116, in _find_ffmpeg_extension
    ext = _find_versionsed_ffmpeg_extension(ffmpeg_ver)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv312\Lib\site-packages\torio\_extension\utils.py", line 108, in _find_versionsed_ffmpeg_extension
    _load_lib(lib)
  File "c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv312\Lib\site-packages\torio\_extension\utils.py", line 94, in _load_lib
    torch.ops.load_library(path)
  File "c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv312\Lib\site-packages\torch\_ops.py", line 1357, in load_library
    ctypes.CDLL(path)
  File "C:\Users\RobynPfeifer\AppData\Roaming\uv\python\cpython-3.12.12-windows-x86_64-none\Lib\cty

In [5]:
# attempt to automatically determine optimal whisperx config based on system specs

# noteworthy config settings to tweak as needed: # TODO add these to the GlobalAppSettings object
prefer_accuracy = True  # for details for both of these params, see docstring for the auto_configure_whisperx function below
prefer_speed = False

# this entire chunk started as pseudocode notes and a skeleton from Claude, and has been mangled enough to where I can't even recognize the original code anymore, likely needs a solid cleanup and refactor, move imports to top of file, etc etc
import os
import sys
import platform
import psutil
import subprocess
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class WhisperXConfig:
    model_size: str
    device: str
    compute_type: str
    batch_size: int
    device_index: int = 0
    threads: int = 4

    # Metadata for debugging/logging, very useful
    _reason: dict = field(default_factory=dict, repr=False)

    def to_dict(self) -> dict:
        return {
            "model_size": self.model_size,
            "device": self.device,
            "compute_type": self.compute_type,
            "batch_size": self.batch_size,
            "device_index": self.device_index,
            "threads": self.threads,
        }

    def explain(self) -> str:
        lines = ["WhisperX Config Rationale:"]
        for k, v in self._reason.items():
            lines.append(f"  {k}: {v}")
            
        return "\n".join(lines)


def _get_cuda_info() -> tuple[bool, Optional[int], Optional[str]]:
    """
    Returns (cuda_available, vram_mb, gpu_name).
    Tries torch first, falls back to nvidia-smi.
    """

    # Try PyTorch first...
    try:
        import torch

        if torch.cuda.is_available():
            idx = torch.cuda.current_device()
            vram = torch.cuda.get_device_properties(idx).total_memory // (1024**2)
            name = torch.cuda.get_device_name(idx)
            return True, vram, name

    except ImportError:
        # not handling here, just letting execution continue and return False etc
        pass

    # Fallback: trying nvidia-smi...
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.total,name",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=5,
        )

        if result.returncode == 0:
            parts = result.stdout.strip().split(",")
            vram = int(parts[0].strip())
            name = parts[1].strip() if len(parts) > 1 else "Unknown GPU"
            return True, vram, name

    except (FileNotFoundError, subprocess.TimeoutExpired, ValueError) as e:
        # not worrying about these exceptions here, just letting execution continue and return False etc
        pass

    return False, None, None


def _get_ram_gb() -> float:
    return psutil.virtual_memory().total / (1024**3)


def _get_cpu_cores() -> int:
    return (
        psutil.cpu_count(logical=False) or psutil.cpu_count(logical=True) or 2
    )  # cascading fallback to default of 2 cores


def _supports_float16_cpu() -> bool:
    """Check if CPU supports float16 via AVX2 (rough heuristic)."""

    try:
        import subprocess

        if platform.system() == "Linux":
            result = subprocess.run(
                ["grep", "-m1", "avx2", "/proc/cpuinfo"], capture_output=True, text=True
            )
            return "avx2" in result.stdout
        
        elif platform.system() == "Darwin":
            result = subprocess.run(
                ["sysctl", "-n", "machdep.cpu.features"], capture_output=True, text=True
            )
            
            return "AVX2" in result.stdout

    except Exception as e:
        # not handling here, just letting execution continue and return False etc
        pass

    return False


def auto_configure_whisperx(
    prefer_accuracy: bool = True,
    prefer_speed: bool = False,
    verbose: bool = False,
) -> WhisperXConfig:
    """
    Attempts to automatically determine optimal WhisperX configuration based on current system specs.

    Args:
        prefer_accuracy: Bias toward larger models even at cost of speed.
        prefer_speed:    Bias toward smaller models / faster compute types.
        verbose:         Print rationale to stdout. (highly suggested)

    Returns:
        WhisperXConfig dataclass with recommended default WhisperX model settings based on system specs.
    """

    reasons = {}

    # ── Hardware detection ───────────────────────────────────────────────────
    cuda_available, vram_mb, gpu_name = _get_cuda_info()
    ram_gb = _get_ram_gb()
    cpu_cores = _get_cpu_cores()
    cpu_float16 = _supports_float16_cpu()

    reasons["ram_gb"] = f"{ram_gb:.1f} GB"
    reasons["cpu_cores"] = cpu_cores
    reasons["cuda"] = (
        f"{cuda_available} ({gpu_name}, {vram_mb} MB VRAM)"
        if cuda_available
        else "not available"
    )

    # ── Device ───────────────────────────────────────────────────────────────
    if cuda_available and vram_mb is not None and vram_mb >= 2048:
        device = "cuda"
        reasons["device"] = "CUDA selected (GPU detected with sufficient VRAM)"
    else:
        device = "cpu"
        reasons["device"] = "CPU selected (no CUDA or insufficient VRAM)"

    # ── Model size ───────────────────────────────────────────────────────────
    # Reference:
    # Approximate VRAM / RAM requirements per model:
    #   tiny   ~1 GB   large-v2 ~10 GB
    #   base   ~1 GB   large-v3 ~10 GB
    #   small  ~2 GB   turbo    ~6 GB
    #   medium ~5 GB
    if device == "cuda":
        resource = vram_mb / 1024  # GB
        resource_label = f"{resource:.1f} GB VRAM"
    else:
        resource = ram_gb
        resource_label = f"{resource:.1f} GB RAM"

    # Apply bias modifiers
    bias = 1 if prefer_accuracy else (-1 if prefer_speed else 0)

    model_tiers = [
        (1.5, "tiny"),
        (2.5, "base"),
        (4.0, "small"),
        (7.0, "medium"),
        (11.0, "large-v2"),
        (
            float("inf"),
            "large-v3",
        ),  # for anything above the 11.0 threshold, use large-v3
    ]

    chosen_model = "tiny"
    for threshold, name in model_tiers:
        if resource >= threshold:
            chosen_model = name

    # Apply bias: shift one tier up or down
    model_names = [m for _, m in model_tiers]
    idx = model_names.index(chosen_model)
    idx = max(0, min(len(model_names) - 1, idx + bias))
    chosen_model = model_names[idx]

    reasons["model_size"] = f"{chosen_model} (based on {resource_label})"

    # ── Compute type ─────────────────────────────────────────────────────────
    if device == "cuda":
        # float16 is standard on modern NVIDIA; use int8 only on very low VRAM
        if vram_mb >= 4096:
            compute_type = "float16"
            reasons["compute_type"] = "float16 (CUDA, ≥4 GB VRAM)"
        else:
            compute_type = "int8_float16"
            reasons["compute_type"] = "int8_float16 (CUDA, limited VRAM)"
    else:
        # CPU: int8 is fastest and well-supported; float16 rarely beneficial on CPU
        if prefer_accuracy and cpu_float16:
            compute_type = "float32"
            reasons["compute_type"] = "float32 (CPU, accuracy mode, AVX2 present)"
        else:
            compute_type = "int8"
            reasons["compute_type"] = "int8 (CPU default — best speed/memory tradeoff)"

    # ── Batch size ───────────────────────────────────────────────────────────
    if device == "cuda":
        if vram_mb >= 10000:
            batch_size = 32
        elif vram_mb >= 6000:
            batch_size = 16
        elif vram_mb >= 4000:
            batch_size = 8
        else:
            batch_size = 4
    else:
        # CPU batch sizing based on RAM and cores
        if ram_gb >= 32 and cpu_cores >= 8:
            batch_size = 8
        elif ram_gb >= 16 and cpu_cores >= 4:
            batch_size = 4
        else:
            batch_size = 2

    if prefer_speed:
        batch_size = min(batch_size * 2, 64)
        reasons["batch_size"] = f"{batch_size} (speed-biased, doubled)"
    elif prefer_accuracy:
        batch_size = max(batch_size // 2, 1)
        reasons["batch_size"] = f"{batch_size} (accuracy-biased, halved)"
    else:
        reasons["batch_size"] = str(batch_size)

    # ── CPU threads ──────────────────────────────────────────────────────────
    # WhisperX passes this to faster-whisper / CTranslate2
    threads = max(2, min(cpu_cores, 8))  # cap at 8; diminishing returns beyond that
    reasons["threads"] = f"{threads} (of {cpu_cores} physical cores)"

    # ── Build config ─────────────────────────────────────────────────────────
    config = WhisperXConfig(
        model_size=chosen_model,
        device=device,
        compute_type=compute_type,
        batch_size=batch_size,
        device_index=0,
        threads=threads,
        _reason=reasons,
    )

    if verbose:
        logging.info(config.explain())

    return config


In [6]:
whisperx_cfg = auto_configure_whisperx(
    verbose=True,
    prefer_speed=prefer_speed,
    prefer_accuracy=prefer_accuracy
)

logging.info(
    f"WhisperX config has been automatically set based on current system specs!"
)


INFO:root:WhisperX Config Rationale:
  ram_gb: 7.7 GB
  cpu_cores: 6
  cuda: not available
  device: CPU selected (no CUDA or insufficient VRAM)
  model_size: large-v2 (based on 7.7 GB RAM)
  compute_type: int8 (CPU default — best speed/memory tradeoff)
  batch_size: 1 (accuracy-biased, halved)
  threads: 6 (of 6 physical cores)
INFO:root:WhisperX config has been automatically set based on current system specs!


In [7]:
# user configurable settings part deux
# most are automatically set via auto_configure_whisperx() above based on system specs, but can be overridden manually here
# change these at your own risk!

logging.info(f"Executing any manual overrides to auto-configured WhisperX model settings...")

MODEL_SIZE = (
    whisperx_cfg.model_size
) # Whisper model: tiny | base | small | medium | large-v2 | large-v3
LANGUAGE = None  # 2 digit ISO code for spoken language in input audio file (without locale code) such as "en" for English. Set to None for auto-detect # note: WhisperX model needs >30s of audio to even attempt to detect the language, and has major difficulties with multilingual audio, compounded by constraint of enforced lack of internet access and local-only model usage (after initial download of cached models)
DEVICE = whisperx_cfg.device # "cpu" or "cuda"
COMPUTE_TYPE = (
    whisperx_cfg.compute_type
)  # "int8" (CPU-friendly) | "float16" (GPU) | "float32"
BATCH_SIZE = 16 # Reduce if you hit OOM on GPU; safe to leave as-is for CPU
ALIGN_OUTPUT = ( # unneeded at this point, resource intensive and time/compute/token usage noticably goes up
    False  # Word-level alignment (requires internet access for first run to download and cache models locally)
)
THREADS = whisperx_cfg.threads # Number of threads to use for transcription
DEVICE_INDEX = whisperx_cfg.device_index # Index of GPU device to use


HF_TOKEN = settings.hf_token  # HuggingFace token — only needed for speaker diarization gated model (located in the .env file in project root)
DIARIZE = True  # Speaker diarization (requires HF_TOKEN set)


INFO:root:Executing any manual overrides to auto-configured WhisperX model settings...


In [8]:
logging.info(f"Configuring path for FFMpeg and determining installed FFMpeg version...")

# attempts to get the version of the ffmpeg executable, assuming it's found in the sys.PATH somewhere
def get_ffmpeg_version() -> str:
    """
    Returns the ffmpeg version string (e.g., '6.1.1').

    Raises:
        RuntimeError: if ffmpeg is not found in path, not installed or version cannot be parsed.
    """

    try:
        result = subprocess.run(
            ["ffmpeg", "-version"], check=True, capture_output=True, text=True
        )

    except FileNotFoundError as e:
        logging.error(
            f"\033[31m ERROR: ffmpeg is not installed or not found in PATH: \033[0m {e}"
        )
        raise RuntimeError("ffmpeg is not installed or not found in PATH.")

    except subprocess.CalledProcessError as e:
        logging.error(f"\033[31m ERROR: ffmpeg returned an error: \033[0m {e}")
        raise RuntimeError(f"ffmpeg returned an error: {e.stderr}") from e

    # Typical first line looks like:
    # "ffmpeg version n6.1.1-3-g123abc blah blah stuff this version was built with, blah blah blah"
    # second and onward lines have build info and other details, not needed for our purposes
    first_line = result.stdout.splitlines()[0]

    match = re.search(r"ffmpeg version\s+([^\s]+)", first_line)
    if not match:
        logging.error(
            f"\033[31m ERROR: Could not parse ffmpeg version from output's first line: \033[0m \"{first_line}\""
        )
        raise RuntimeError("Could not parse ffmpeg version output.")

    version = match.group(1)

    # Optional: strip leading 'n' sometimes present in builds (e.g., n6.1.1)
    version = version.lstrip("n")

    return version


def print_formatted_sys_path():
    # Rather than deal with the raw output of sys.PATH, we can use the json module to spit out a nicely formatted list of the current sys.PATHs
    formatted_path = json.dumps(sys.path, indent=4)

    # use some colors to make it pretty!
    # not gonna lie, I asked Claude for the ANSI codes for changing terminal output color, without shame. This simple use didn't warrant importing a full proper terminal color package like rich or colorama, etc
    logging.info("\033[1;34m [sys.PATH] \033[0m")  # ANSI code for blue for the title
    logging.info(
        "\033[1;32m" + formatted_path + "\033[0m"
    )  # ANSI code for green for paths, then ANSI code for reset


if DEBUG_MODE:
    logging.info(f"Outputting current sys.PATH:")
    print_formatted_sys_path()

try:
    ffmpeg_version = get_ffmpeg_version()

except Exception as e:
    logging.error(f"\033[31m General Recoverable ERROR: Could not find ffmpeg version: \033[0m {e}")
    # this is easily recoverable, so I'm choosing to not re-raise an exception, but instead just print it, and default to adding the local ffmpeg bin path to sys.PATH
    # raise RuntimeError("ffmpeg not found in path.")

    # ffmpeg not found in path => add local ffmpeg path (in project root) to path
    logging.info(f"Attempting to add local ffmpeg path to sys.PATH...")
    parent_dir = Path.cwd().resolve().parent  # parent = project root, currently running from test_notebooks dir, one folder down from project root, will need update when merged to main # TODO update this when merged to main
    ffmpeg_bin_path = (
        parent_dir / "ffmpeg" / "bin"
    )  # will look for local ffmpeg here, as well as the rest of sys.PATH

    if ffmpeg_bin_path.exists():
        sys.path.append(str(ffmpeg_bin_path))
        logging.info(
            f"Directory `{ffmpeg_bin_path}` found and has been added to sys.PATH!"
        )

        if DEBUG_MODE:
            logging.info(
                f"Outputting freshly updated sys.PATH to ensure our added path is properly set in the sys.PATH:"
            )
            print_formatted_sys_path()

logging.info(
    f"Success: FFMpeg executable found in sys.PATH! Installed FFMpeg version: {ffmpeg_version}"
)


INFO:root:Configuring path for FFMpeg and determining installed FFMpeg version...
INFO:root:Outputting current sys.PATH:
INFO:root: [sys.PATH] 
INFO:root:[
    "C:\\Users\\RobynPfeifer\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\python312.zip",
    "C:\\Users\\RobynPfeifer\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\DLLs",
    "C:\\Users\\RobynPfeifer\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib",
    "C:\\Users\\RobynPfeifer\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none",
    "c:\\Users\\RobynPfeifer\\code\\STAT405_AudioNotetaker\\.venv312",
    "",
    "c:\\Users\\RobynPfeifer\\code\\STAT405_AudioNotetaker\\.venv312\\Lib\\site-packages"
]
INFO:root:Success: FFMpeg executable found in sys.PATH! Installed FFMpeg version: N-122750-g6ee3e59ce2-20260215


In [9]:
# ensure folders and paths exist
# the models_dir is automatically created regardless, and as of now, output is dumped in cwd, to change later
# sample_data folder doesn't exactly need to be tested for existence for now


In [10]:
setup_end_time = time.perf_counter()  # end time for setup
setup_duration = setup_end_time - start_time  # in secs, reusing start_time as time marker for the beginning of the setup phase
setup_duration_str = str(timedelta(seconds=setup_duration))  # convert to stringified timedelta HH:MM:SS.mmm format

logging.info(
    f"Imports, initial setup, FFMpeg detection, CUDA detection, hardware spec detection, and remaining configuration options completed in {setup_duration_str}! Current datetime: {datetime.now()}"
)

# next several chunks are mostly wrappers and helpers to wrangle the WhisperX model and associated fucntionality


INFO:root:Imports, initial setup, FFMpeg detection, CUDA detection, hardware spec detection, and remaining configuration options completed in 0:00:21.043231! Current datetime: 2026-03-09 03:56:35.016060


In [11]:
# setting up the whisperx model, just a simple wrapper
def load_model(model_size: str, device: str, compute_type: str) -> WhisperModel:
    """Load the WhisperX ASR model."""
    
    logging.info(
        f"[1/4] Loading Whisper model '{model_size}' on {device} ({compute_type})..."
    )
    
    model = whisperx.load_model(
        model_size,
        device=device,
        compute_type=compute_type,
        language=LANGUAGE,
        download_root=MODEL_DIR,
    )
    
    return model


In [12]:
# bypassing all the issues with torchcodec, using ffmpeg instead to decode audio
def load_audio_via_ffmpeg(file_path: str, sample_rate: int = 16000) -> dict:
    """
    Decode audio using ffmpeg -> numpy -> torch tensor process flow.
    Returns a dict that pyannote/whisperx can accept as pre-loaded audio in format:
        {'waveform': (1, N) torch.Tensor, 'sample_rate': int}
    Bypasses torchcodec entirely. Excellent.
    """

    # don't question the black magic voodoo below, just let ffmpeg do its thing and hand it off to pytorch
    cmd = [
        "ffmpeg",
        "-nostdin",
        "-threads",
        "0",
        "-i",
        file_path,
        "-f",
        "f32le",  # raw 32-bit float PCM, little-endian
        "-ac",
        "1",  # mono
        "-ar",
        str(sample_rate),  # resample to target rate
        "-",  # pipe to stdout
    ]

    result = subprocess.run(cmd, capture_output=True, check=True)
    audio_np = np.frombuffer(result.stdout, dtype=np.float32).copy()  # deep copy
    waveform = torch.from_numpy(audio_np).unsqueeze(0)  # (1, N)

    return {"waveform": waveform, "sample_rate": sample_rate}


In [13]:
# trying to get anything to callback with progress updates has so far been a complete failure, though the verbose/print_progress flags has helped somewhat
def progress_callback(percent_complete):
    """A simple callback function to output transcription progress updates."""
    
    logging.info(f"Transcription Progress Update: {percent_complete}% completed!")


In [14]:
# largely a simple wrapper, but this is the core function right here
def transcribe_audio(model, audio_path: str) -> tuple[dict, dict]:
    """Run the initial transcription pass using the ffmpeg-decoded audio."""

    logging.info(f"[2/4] Transcribing '{audio_path}' ...")
    t0 = time.time()

    audio_input = load_audio_via_ffmpeg(audio_path)

    # whisperx.transcribe also accepts the raw numpy array (1-D float32)
    audio_np = audio_input["waveform"].squeeze(0).numpy()

    result = model.transcribe(
        audio_np,
        batch_size=BATCH_SIZE,
        language=LANGUAGE,
        print_progress=True, # progress bar!
    )

    logging.info(
        f"      Done in {time.time() - t0:.1f}s  |  detected language: {result.get('language', 'unknown')}"
    )

    # Return both result and raw audio_input dict (needed for later alignment/diarization steps)
    return result, audio_input


In [15]:
# word level timestamp alignment
def align_transcript(result: dict, audio: np.ndarray, device: str) -> dict:
    """Align transcript to audio for word-level timestamps."""

    logging.info("[3/4] Aligning transcript (word-level timestamps) ...")
    lang = result.get("language", "en")  # defaults to english if unable to auto-detect

    try:
        align_model, align_metadata = whisperx.load_align_model(
            language_code=lang,
            device=device,
        )

        result_aligned = whisperx.align(
            result["segments"],
            align_model,
            align_metadata,
            audio,
            device,
            return_char_alignments=False,
        )

        return result_aligned

    except Exception as e:
        logging.error(
            f"      \033[31m Recoverable ERROR! Alignment failed: \033[0m {e}. \nDefaulting to using unaligned output."
        )

        return result


In [16]:
# the hardest bit, diarization of the transcribed audio...
def diarize_transcript(
    result: dict, audio_input: dict, hf_token: str, device: str
) -> dict:
    """Assign speaker labels via pyannote diarization."""

    logging.info(f"[3b/4] Starting diarization (segment speaker identification) phase...")

    try:
        from pyannote.audio import Pipeline

        diarize_model = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            token=hf_token,
            cache_dir=MODEL_DIR
        ).to(torch.device(device))

        diarized_segments = diarize_model(audio_input)

        result = whisperx.assign_word_speakers(diarized_segments, result)

    except Exception as e:
        logging.error(
            f"      \033[31m Recoverable ERROR: Diarization failed: \033[0m {e}. Skipping diarization..."
        )

    return result


In [17]:
# manipulate default timestamp format here (for transcript timestamps only!):
def format_timestamp(seconds: float) -> str:
    """Convert float seconds to HH:MM:SS.mmm string."""

    ms = int((seconds % 1) * 1000)
    s = int(seconds)
    m, s = divmod(s, 60)
    h, m = divmod(m, 60)

    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"


In [ ]:
# basic transcript output until diarization problem is solved, modify this as needed for desired style of output transcript # TODO may be worth duplicating this function somewhat and/or adding a parameter for output_type to return the transcript in a format that is more easily parsed by other parts of the pipeline downstream (JSON probably is best bet?), whatever form it may take
def build_transcript_text(result: dict) -> str:
    """
    Render speaker segments to a readable plain-text transcript.
    Default format per line:
        [HH:MM:SS.mmm --> HH:MM:SS.mmm]  (SPEAKER_XX)  text
    """

    lines = []

    for seg in result.get("segments", []):
        start = format_timestamp(seg.get("start", 0.0))
        end = format_timestamp(seg.get("end", 0.0))
        text = seg.get("text", "").strip()
        speaker = seg.get("speaker", "")
        speaker_tag = f"  [{speaker}]" if speaker else ""

        lines.append(f"[{start} --> {end}]{speaker_tag}  {text}")

    return "\n".join(lines)


In [ ]:
def save_output(transcript_text: str, result: dict, output_path: str) -> None:
    """Write formatted transcript and a companion JSON file with properly formatted metadata."""

    logging.info(f"[4/4] Writing transcript to '{output_path}'...")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(transcript_text)
        f.write("\n")  # trailing newline

    # simultaneously save full JSON dump to file (with timestamps, word data, etc) alongside the txt, same filename stem, but with .json extension
    json_path = os.path.splitext(output_path)[0] + ".json"

    # TODO add additional metadata to JSON object before writing to file/object in memory
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    logging.info(f"      JSON data saved to '{json_path}'")


In [20]:
# --- Validate input ---
if not os.path.isfile(INPUT_FILE):
    logging.error(f"\033[31m FATAL ERROR: Input file not found: \033[0m '{INPUT_FILE}'!")
    sys.exit(1)  # fatality: unrecoverable, abort

# --- Pipeline ---
model_load_start_time = time.perf_counter()  # start time for model loading (to measure download speed performance in online mode, and attempt to gauge the speed at which it can actually load the cached local model in offline mode)
model = load_model(MODEL_SIZE, DEVICE, COMPUTE_TYPE)
model_load_end_time = time.perf_counter()  # end time for model loading
model_load_duration_str = str(timedelta(seconds = model_load_end_time - model_load_start_time))
logging.info(f"Initial WhisperX model loaded in {model_load_duration_str}.")


transcription_start_time = time.perf_counter()  # start time for transcription
(result, audio_input) = transcribe_audio(model, INPUT_FILE)
transcription_end_time = time.perf_counter()  # end time for transcription
transcription_duration_str = str(timedelta(seconds = transcription_end_time - transcription_start_time))
logging.info(f"Audio transcription/translation phase completed in {transcription_duration_str}!")


logging.info(f"Freeing up memory by releasing current model from RAM/VRAM...")

# Free model memory before loading align model to conserve already-scarce hardware resources (very helpful on CPU/low-RAM, on low-spec'd devices, this is basically 100% necessary)
del model  # this is super duper important running on mediocre hardware

# word aligned output
if ALIGN_OUTPUT:
    alignment_start_time = time.perf_counter()  # start time for alignment
    result = align_transcript(
        result, audio_input["waveform"].squeeze(0).numpy(), DEVICE
    )
    alignment_end_time = time.perf_counter()  # end time for alignment
    alignment_duration_str = str(timedelta(seconds = alignment_end_time - alignment_start_time))
    logging.info(f"Alignment process completed in {alignment_duration_str}!")

# diarization (segment speaker identification and labeling)
if DIARIZE:
    if not HF_TOKEN: # needs valid HF token to initially download the gated models, after that, can run off cached models offline
        logging.warning(
            f"\033[31m WARNING: DIARIZE=True but HF_TOKEN is not set! \033[0m Skipping diarization process..."
        )

    else:
        logging.info(f"Beginning diarization phase...")
        diarization_start_time = time.perf_counter()  # start time for diarization
        result = diarize_transcript(result, audio_input, HF_TOKEN, DEVICE)
        diarization_end_time = time.perf_counter()  # end time for diarization
        diarization_duration_str = str(timedelta(seconds = diarization_end_time - diarization_start_time))
        logging.info(f"Diarization process completed in {diarization_duration_str}!")


# the following bit will be removed once this code is merged into main, as we won't be writing to a local data file (unnecessary), we can simply keep the transcript JSON object in memory to hand off to the next step in the pipeline
build_and_save_transcript_start_time = time.perf_counter()  # start time for building and saving transcript files to local store
transcript_text = build_transcript_text(result)
save_output(transcript_text, result, OUTPUT_FILE)
build_and_save_transcript_end_time = time.perf_counter()  # end time for building and saving transcript files
build_and_save_transcript_duration_str = str(timedelta(seconds = build_and_save_transcript_end_time - build_and_save_transcript_start_time))
logging.info(f"Building the transcript text from the JSON object and export of data to local txt/json files completed in {build_and_save_transcript_duration_str}!")


logging.info(f"Transcription process fully complete! Current datetime: {datetime.now()}")
logging.info(f"  Text : {OUTPUT_FILE}")
logging.info(f"  JSON : {os.path.splitext(OUTPUT_FILE)[0]}.json")

# despite multiple attempts to suppress the Lightning checkpoint version auto-upgrade part, as well as running the (supposed "permanent" fix) upgrade command as the output suggests, it yet persists... As it says, it automatically upgrades the checkpoint version by itself, so this can safely be ignored and written off as additional annoyingly verbose output


INFO:root:[1/4] Loading Whisper model 'large-v2' on cpu (int8)...


2026-03-09 03:57:21 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-03-09 03:57:21 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


DEBUG:fsspec.local:open file: c:/Users/RobynPfeifer/code/STAT405_AudioNotetaker/.venv312/Lib/site-packages/whisperx/assets/pytorch_model.bin
DEBUG:fsspec.local:open file: c:/Users/RobynPfeifer/code/STAT405_AudioNotetaker/.venv312/Lib/site-packages/whisperx/assets/pytorch_model.bin
INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv312\Lib\site-packages\whisperx\assets\pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint c:\Users\RobynPfeifer\code\STAT405_AudioNotetaker\.venv312\Lib\site-packages\whisperx\assets\pytorch_model.bin`
INFO:root:Initial WhisperX model loaded in 0:00:49.325046.
IN

2026-03-09 03:58:01 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (9): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (10): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (11): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (12): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (13): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (14): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (15): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (16): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (17): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (18): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (19): otel.pyannote.ai:443
DEBUG:urllib3.connectionpool:Starting new HTTPS connect

Progress: 11.11%...
Progress: 22.22%...
Progress: 33.33%...
Progress: 44.44%...
Progress: 55.56%...
Progress: 66.67%...
Progress: 77.78%...
Progress: 88.89%...
Progress: 100.00%...


INFO:root:Beginning diarization phase...
INFO:root:[3b/4] Starting diarization (segment speaker identification) phase...
DEBUG:fsspec.local:open file: c:/Users/RobynPfeifer/code/STAT405_AudioNotetaker/test_notebooks/models/models--pyannote--segmentation-3.0/snapshots/e66f3d3b9eb0873085418a7b813d3b369bf160bb/pytorch_model.bin
DEBUG:fsspec.local:open file: c:/Users/RobynPfeifer/code/STAT405_AudioNotetaker/test_notebooks/models/models--pyannote--segmentation-3.0/snapshots/e66f3d3b9eb0873085418a7b813d3b369bf160bb/pytorch_model.bin
DEBUG:fsspec.local:open file: c:/Users/RobynPfeifer/code/STAT405_AudioNotetaker/test_notebooks/models/models--pyannote--wespeaker-voxceleb-resnet34-LM/snapshots/837717ddb9ff5507820346191109dc79c958d614/pytorch_model.bin
DEBUG:fsspec.local:open file: c:/Users/RobynPfeifer/code/STAT405_AudioNotetaker/test_notebooks/models/models--pyannote--wespeaker-voxceleb-resnet34-LM/snapshots/837717ddb9ff5507820346191109dc79c958d614/pytorch_model.bin
c:\Users\RobynPfeifer\code\

In [ ]:
end_time = time.perf_counter()  # end time for execution
logging.info(f"Execution completed! End time: {datetime.now()}")

execution_duration = end_time - start_time  # in secs
duration_str = str(
    timedelta(seconds=execution_duration)
)  # convert to stringified timedelta HH:MM:SS.mmm format

logging.info(f"Entire (transcription/translation/diarization/alignment steps of the pipeline) process pipeline completed in {duration_str}!")

# initial testing on this lacking hardware laptop, with diarization and alignment both enabled, with online mode enabled to download and cache the required models, using an uncompressed WAV file (duration: ~3:17) in testing resulted in a non-diarized (due to failure in pipeline) word-aligned transcript in approximately 21 minutes. Utilizing that as a rough benchmark, a user with a similarly-powered device would likely expect to see a similar processing time roughly proportional to the duration of the input audio file, approximately 7:1 (in this specific case)in terms of ratio of processing time to initial audio file length. Subsequent testing has been inconsistent and all over the map, highly reactive towards _any_ other applications running that require even tiny amounts of resources, ie. no browsers, no additional VSCode instances, etc. For lowly-spec'd devices, would suggest running this application after a fresh reboot.
# additional tests with various other test audio files (mp4, mp3, etc) resulted in similar computation/processing times.
# projecting from here, a "typical" 50-minute "hour" therapy session would take at minimum, 350 minutes (5 hours, 50 minutes) to transcribe as a lower bound, and many of my failing test files were unable to be diarized properly, so it's a reasonable assumption that we're looking at a baseline minimum processing time of _at least_ ~6 hours for a single 50-minute therapy session when the diarization pipeline is used as well, as in the "normal" use case we've primarily designed for. Despite this significantly lackluster performance, the effect of this level of processing/compute time will be minimized by processing the audio files in the background, and/or to be processed overnight, etc.
# Additional hits to performance likely would be caused by things such as varying (as in multiple languages present in the same audio file) languages used in the input audio, as well as increasing complexity _significantly_ if there are multiple speakers (the diarization pipeline, given our hardware constraints, does not do so great at this), possibly requiring further pre-processing of the audio file into single line or single-language segments before sending off the data down the rest of the processing pipeline. This is attainable, but would complicate things somewhat, and would likely also result in a performance hit due to trading a single (or very few, depending on batch size configuration, etc) call(s) to each model for the entire file for calling a model _for each and every_ segmented line in the file. Current testing results are not promising if the input has multiple languages spoken, significantly more so if each segmented line can be multilingual, as in the case of a native speaker using common cultural expressions or slang in their native language in a single segment/spoken line. This would likely manifest as a moderately-involved refactoring of the pipeline to facilitate this, but would also likely result in a noticeable performance hit by making so many more calls to the resource intensive models.
# performance could be improved by opting to _not_ use the alignment pipeline, and foregoing word-level timestamp alignment, as such detailed identification is not necessarily required for this project.

# language detection accuracy, transcription accuracy, and translation accuracy measures have not yet been implemented as of yet. 


INFO:root:Execution completed! End time: 2026-03-09 04:31:44.373636
INFO:root:Entire (transcription/translation/diarization/alignment steps of the pipeline) process pipeline completed in 0:35:30.541185!
